<a href="https://colab.research.google.com/github/NyxLumen/Encephlo/blob/main/Notebooks/Tri_Train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!unzip -q cleaned_data.zip -d dataset/

In [5]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

# ==========================================
# 1. HARDWARE & DATA MOUNTING
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🔥 NEURAL ENGINE MOUNTED ON: {device}")

clinical_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

TRAIN_DIR = 'dataset/cleaned_data/training'
VAL_DIR = 'dataset/cleaned_data/testing'

if not os.path.exists(TRAIN_DIR) or not os.path.exists(VAL_DIR):
    print(f"❌ ERROR: Cannot find data directories. Check your unzipped folder structure.")
else:
    print("✅ DATASET LOCATED. Mounting loaders...")

train_data = datasets.ImageFolder(TRAIN_DIR, transform=clinical_transform)
val_data = datasets.ImageFolder(VAL_DIR, transform=clinical_transform)

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader = DataLoader(val_data, batch_size=32, shuffle=False)

# ==========================================
# 2. THE RESEARCH TRAINING ENGINE
# ==========================================
def train_model(model, name, epochs=30, patience=7):
    print(f"\n🚀 INITIATING 30-EPOCH FORGE: {name} (Early Stopping Patience: {patience})")
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-4)

    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

    best_val_loss = float('inf')
    epochs_no_improve = 0
    actual_epochs_run = 0

    for epoch in range(epochs):
        actual_epochs_run += 1

        # --- TRAINING PHASE ---
        model.train()
        train_loss, correct_train, total_train = 0.0, 0, 0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            _, predicted = outputs.max(1)
            total_train += labels.size(0)
            correct_train += predicted.eq(labels).sum().item()

        epoch_train_loss = train_loss / len(train_loader)
        epoch_train_acc = 100. * correct_train / total_train

        # --- VALIDATION PHASE ---
        model.eval()
        val_loss, correct_val, total_val = 0.0, 0, 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)

                val_loss += loss.item()
                _, predicted = outputs.max(1)
                total_val += labels.size(0)
                correct_val += predicted.eq(labels).sum().item()

        epoch_val_loss = val_loss / len(val_loader)
        epoch_val_acc = 100. * correct_val / total_val

        history['train_loss'].append(epoch_train_loss)
        history['val_loss'].append(epoch_val_loss)
        history['train_acc'].append(epoch_train_acc)
        history['val_acc'].append(epoch_val_acc)

        print(f"Epoch {epoch+1:02d}/{epochs} | "
              f"Train Loss: {epoch_train_loss:.4f} (Acc: {epoch_train_acc:.2f}%) | "
              f"Val Loss: {epoch_val_loss:.4f} (Acc: {epoch_val_acc:.2f}%)")

        # --- CHECKPOINT & EARLY STOPPING ---
        if epoch_val_loss < best_val_loss:
            best_val_loss = epoch_val_loss
            epochs_no_improve = 0
            torch.save(model.state_dict(), f"{name}_best_weights.pth")
            print(f"   🌟 CHECKPOINT: New best model saved! (Val Loss: {best_val_loss:.4f})")
        else:
            epochs_no_improve += 1
            print(f"   ⚠️ No improvement for {epochs_no_improve} epoch(s).")

        if epochs_no_improve >= patience:
            print(f"\n🛑 EARLY STOPPING TRIGGERED AT EPOCH {epoch+1}. Model converged.")
            break

    # --- GENERATE IEEE GRAPHS ---
    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    plt.plot(range(1, actual_epochs_run + 1), history['train_loss'], label='Train Loss', color='blue')
    plt.plot(range(1, actual_epochs_run + 1), history['val_loss'], label='Validation Loss', color='red')
    plt.title(f'{name} - Loss Curve')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.6)

    plt.subplot(1, 2, 2)
    plt.plot(range(1, actual_epochs_run + 1), history['train_acc'], label='Train Accuracy', color='blue')
    plt.plot(range(1, actual_epochs_run + 1), history['val_acc'], label='Validation Accuracy', color='red')
    plt.title(f'{name} - Accuracy Curve')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy (%)')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.6)

    plt.tight_layout()
    plt.savefig(f"{name}_training_graphs.png", dpi=300)
    plt.close()
    print(f"📊 {name} GRAPHS GENERATED AND SAVED.")

    return model

# ==========================================
# 3. MODEL ASSEMBLY & EXECUTION
# ==========================================
print("\n🧬 ASSEMBLING DENSENET121...")
densenet = models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)
densenet.classifier = nn.Sequential(nn.Dropout(0.5), nn.Linear(densenet.classifier.in_features, 4))
train_model(densenet, "densenet121", epochs=30, patience=7)

print("\n🧬 ASSEMBLING EFFICIENTNET-B0...")
effnet = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
effnet.classifier[1] = nn.Linear(effnet.classifier[1].in_features, 4)
train_model(effnet, "efficientnet_b0", epochs=30, patience=7)

print("\n👁️ ASSEMBLING VISION TRANSFORMER (ViT-B/16)...")
vit = models.vit_b_16(weights=models.ViT_B_16_Weights.DEFAULT)
vit.heads.head = nn.Linear(vit.heads.head.in_features, 4)
train_model(vit, "vit_b_16", epochs=30, patience=7)

print("\n🏆 MASTER PHASE 1 COMPLETE. All models forged. Checkpoints and Graphs secured.")

🔥 NEURAL ENGINE MOUNTED ON: cuda
✅ DATASET LOCATED. Mounting loaders...

🧬 ASSEMBLING DENSENET121...
Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 176MB/s]



🚀 INITIATING 30-EPOCH FORGE: densenet121 (Early Stopping Patience: 7)
Epoch 01/30 | Train Loss: 0.3248 (Acc: 88.39%) | Val Loss: 0.1211 (Acc: 95.42%)
   🌟 CHECKPOINT: New best model saved! (Val Loss: 0.1211)
Epoch 02/30 | Train Loss: 0.0819 (Acc: 97.65%) | Val Loss: 0.0516 (Acc: 98.40%)
   🌟 CHECKPOINT: New best model saved! (Val Loss: 0.0516)
Epoch 03/30 | Train Loss: 0.0289 (Acc: 99.28%) | Val Loss: 0.0471 (Acc: 98.70%)
   🌟 CHECKPOINT: New best model saved! (Val Loss: 0.0471)
Epoch 04/30 | Train Loss: 0.0264 (Acc: 99.26%) | Val Loss: 0.0333 (Acc: 99.24%)
   🌟 CHECKPOINT: New best model saved! (Val Loss: 0.0333)
Epoch 05/30 | Train Loss: 0.0213 (Acc: 99.44%) | Val Loss: 0.0377 (Acc: 98.70%)
   ⚠️ No improvement for 1 epoch(s).
Epoch 06/30 | Train Loss: 0.0138 (Acc: 99.65%) | Val Loss: 0.0760 (Acc: 97.86%)
   ⚠️ No improvement for 2 epoch(s).
Epoch 07/30 | Train Loss: 0.0215 (Acc: 99.32%) | Val Loss: 0.0612 (Acc: 98.40%)
   ⚠️ No improvement for 3 epoch(s).
Epoch 08/30 | Train Loss: 

100%|██████████| 20.5M/20.5M [00:00<00:00, 211MB/s]


🚀 INITIATING 30-EPOCH FORGE: efficientnet_b0 (Early Stopping Patience: 7)


Epoch 01/30 | Train Loss: 0.4400 (Acc: 86.12%) | Val Loss: 0.1734 (Acc: 92.98%)
   🌟 CHECKPOINT: New best model saved! (Val Loss: 0.1734)
Epoch 02/30 | Train Loss: 0.1294 (Acc: 95.55%) | Val Loss: 0.0935 (Acc: 96.57%)
   🌟 CHECKPOINT: New best model saved! (Val Loss: 0.0935)
Epoch 03/30 | Train Loss: 0.0615 (Acc: 98.28%) | Val Loss: 0.0503 (Acc: 98.25%)
   🌟 CHECKPOINT: New best model saved! (Val Loss: 0.0503)
Epoch 04/30 | Train Loss: 0.0428 (Acc: 98.60%) | Val Loss: 0.0283 (Acc: 98.86%)
   🌟 CHECKPOINT: New best model saved! (Val Loss: 0.0283)
Epoch 05/30 | Train Loss: 0.0245 (Acc: 99.26%) | Val Loss: 0.0247 (Acc: 98.86%)
   🌟 CHECKPOINT: New best model saved! (Val Loss: 0.0247)
Epoch 06/30 | Train Loss: 0.0248 (Acc: 99.21%) | Val Loss: 0.0260 (Acc: 98.86%)
   ⚠️ No improvement for 1 epoch(s).
Epoch 07/30 | Train Loss: 0.0212 (Acc: 99.40%) | Val Loss: 0.0216 (Acc: 99.39%)
   🌟 CHECKPOINT: New best model saved! (Val Loss: 0.0216)
Epoch 08/30 | Train Loss: 0.0155 (Acc: 99.58%) | Val Lo

100%|██████████| 330M/330M [00:07<00:00, 47.5MB/s]



🚀 INITIATING 30-EPOCH FORGE: vit_b_16 (Early Stopping Patience: 7)
Epoch 01/30 | Train Loss: 0.3090 (Acc: 88.38%) | Val Loss: 0.1700 (Acc: 93.97%)
   🌟 CHECKPOINT: New best model saved! (Val Loss: 0.1700)
Epoch 02/30 | Train Loss: 0.1181 (Acc: 95.90%) | Val Loss: 0.0805 (Acc: 97.79%)
   🌟 CHECKPOINT: New best model saved! (Val Loss: 0.0805)
Epoch 03/30 | Train Loss: 0.0487 (Acc: 98.32%) | Val Loss: 0.0898 (Acc: 96.57%)
   ⚠️ No improvement for 1 epoch(s).
Epoch 04/30 | Train Loss: 0.0545 (Acc: 98.13%) | Val Loss: 0.1029 (Acc: 96.26%)
   ⚠️ No improvement for 2 epoch(s).
Epoch 05/30 | Train Loss: 0.0456 (Acc: 98.55%) | Val Loss: 0.2692 (Acc: 91.99%)
   ⚠️ No improvement for 3 epoch(s).
Epoch 06/30 | Train Loss: 0.0439 (Acc: 98.32%) | Val Loss: 0.1009 (Acc: 97.03%)
   ⚠️ No improvement for 4 epoch(s).
Epoch 07/30 | Train Loss: 0.0350 (Acc: 98.81%) | Val Loss: 0.1630 (Acc: 95.04%)
   ⚠️ No improvement for 5 epoch(s).
Epoch 08/30 | Train Loss: 0.0081 (Acc: 99.79%) | Val Loss: 0.0768 (Acc:

In [6]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
from torchvision import models
import torch.nn as nn

# 1. Get the actual class names from your dataset folder structure
class_names = train_data.classes
print(f"🎯 Classes detected: {class_names}")

def generate_confusion_matrix(model, weights_path, name):
    print(f"\n🔍 GENERATING CONFUSION MATRIX FOR: {name}...")

    # Load the absolute best weights we saved during early stopping
    model.load_state_dict(torch.load(weights_path))
    model = model.to(device)
    model.eval()

    all_preds = []
    all_labels = []

    # Run the testing dataset through without updating gradients
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    # Calculate the math
    cm = confusion_matrix(all_labels, all_preds)

    # Plot the IEEE-Ready Heatmap
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names,
                linewidths=1, linecolor='black',
                annot_kws={"size": 14, "weight": "bold"})

    plt.title(f'{name} - Confusion Matrix', fontsize=16, pad=20)
    plt.ylabel('Actual Biological Label', fontsize=12, fontweight='bold')
    plt.xlabel('Model Prediction', fontsize=12, fontweight='bold')
    plt.xticks(rotation=45)
    plt.yticks(rotation=0)

    plt.tight_layout()
    plt.savefig(f"{name}_confusion_matrix.png", dpi=300)
    plt.close()

    print(f"✅ {name}_confusion_matrix.png SAVED TO DISK.")

# ==========================================
# 2. EXECUTE FOR ALL THREE MODELS
# ==========================================

# Reload DenseNet Blueprint
densenet = models.densenet121(weights=None)
densenet.classifier = nn.Sequential(nn.Dropout(0.5), nn.Linear(1024, 4))
generate_confusion_matrix(densenet, "densenet121_best_weights.pth", "DenseNet121")

# Reload EfficientNet Blueprint
effnet = models.efficientnet_b0(weights=None)
effnet.classifier[1] = nn.Linear(1280, 4)
generate_confusion_matrix(effnet, "efficientnet_b0_best_weights.pth", "EfficientNet_B0")

# Reload ViT Blueprint
vit = models.vit_b_16(weights=None)
vit.heads.head = nn.Linear(768, 4)
generate_confusion_matrix(vit, "vit_b_16_best_weights.pth", "ViT_B_16")

print("\n🏆 ALL MATRICES GENERATED. Send them to Overleaf.")

🎯 Classes detected: ['glioma', 'meningioma', 'notumor', 'pituitary']

🔍 GENERATING CONFUSION MATRIX FOR: DenseNet121...
✅ DenseNet121_confusion_matrix.png SAVED TO DISK.

🔍 GENERATING CONFUSION MATRIX FOR: EfficientNet_B0...
✅ EfficientNet_B0_confusion_matrix.png SAVED TO DISK.

🔍 GENERATING CONFUSION MATRIX FOR: ViT_B_16...
✅ ViT_B_16_confusion_matrix.png SAVED TO DISK.

🏆 ALL MATRICES GENERATED. Send them to Overleaf.


In [7]:
import torch
import torch.nn as nn
from torchvision import models
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix, roc_curve, auc
from sklearn.preprocessing import label_binarize
from sklearn.manifold import TSNE
import warnings
warnings.filterwarnings('ignore')

print("⚔️ INITIATING PHASE 2: DECAPITATION & 3072-D FUSION ENGINE...")

# ==========================================
# 1. LOAD & DECAPITATE MODELS
# ==========================================
# We load the blueprints, inject your best weights, and then replace the head with nn.Identity()
# This forces the model to output raw math (features) instead of tumor predictions.

print("🪓 Decapitating DenseNet121 (Extracting 1024-D Vectors)...")
densenet = models.densenet121(weights=None)
densenet.classifier = nn.Sequential(nn.Dropout(0.5), nn.Linear(1024, 4))
densenet.load_state_dict(torch.load("densenet121_best_weights.pth"))
densenet.classifier = nn.Identity()
densenet = densenet.to(device).eval()

print("🪓 Decapitating EfficientNet-B0 (Extracting 1280-D Vectors)...")
effnet = models.efficientnet_b0(weights=None)
effnet.classifier[1] = nn.Linear(1280, 4)
effnet.load_state_dict(torch.load("efficientnet_b0_best_weights.pth"))
effnet.classifier = nn.Identity()
effnet = effnet.to(device).eval()

print("🪓 Decapitating ViT-B/16 (Extracting 768-D Vectors)...")
vit = models.vit_b_16(weights=None)
vit.heads.head = nn.Linear(768, 4)
vit.load_state_dict(torch.load("vit_b_16_best_weights.pth"))
vit.heads.head = nn.Identity()
vit = vit.to(device).eval()

# ==========================================
# 2. FEATURE EXTRACTION (THE FORGE)
# ==========================================
def extract_3072D_features(dataloader, desc=""):
    print(f"🧬 Extracting 3072-D Vectors for {desc}...")
    features_list = []
    labels_list = []

    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs = inputs.to(device)

            # Extract independent vectors
            f_dense = densenet(inputs) # 1024
            f_eff = effnet(inputs)     # 1280
            f_vit = vit(inputs)        # 768

            # Horizontal Concatenation -> 3072-D Vector
            f_fused = torch.cat((f_dense, f_eff, f_vit), dim=1)

            features_list.append(f_fused.cpu().numpy())
            labels_list.append(labels.numpy())

    return np.vstack(features_list), np.concatenate(labels_list)

X_train, y_train = extract_3072D_features(train_loader, "Training Data")
X_test, y_test = extract_3072D_features(val_loader, "Testing Data")

print(f"✅ Extraction Complete. Fused Matrix Shape: {X_train.shape}")

# ==========================================
# 3. THE SUPPORT VECTOR MACHINE (SVM)
# ==========================================
print("\n🧠 TRAINING THE MASTER SVM CLASSIFIER...")
# We use probability=True so we can calculate the ROC-AUC curves
svm = SVC(kernel='rbf', C=1.0, probability=True, random_state=42)
svm.fit(X_train, y_train)

# Generate Predictions
y_pred = svm.predict(X_test)
y_prob = svm.predict_proba(X_test)

final_accuracy = accuracy_score(y_test, y_pred) * 100
print(f"\n🏆 MASTER SVM ACCURACY: {final_accuracy:.2f}%")

# ==========================================
# 4. IEEE DIAGRAM GENERATOR
# ==========================================
print("\n📊 GENERATING IEEE PUBLICATION DIAGRAMS...")
class_names = train_data.classes

# --- DIAGRAM 1: Master Confusion Matrix ---
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=class_names, yticklabels=class_names,
            linewidths=1, linecolor='black', annot_kws={"size": 14, "weight": "bold"})
plt.title(f'Tri-Model SVM Fusion - Confusion Matrix\nAccuracy: {final_accuracy:.2f}%', fontsize=14, pad=15)
plt.ylabel('Actual Biological Label', fontweight='bold')
plt.xlabel('Master SVM Prediction', fontweight='bold')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("SVM_Fusion_Confusion_Matrix.png", dpi=300)
plt.close()

# --- DIAGRAM 2: ROC-AUC Curve ---
y_test_bin = label_binarize(y_test, classes=[0, 1, 2, 3])
n_classes = y_test_bin.shape[1]
fpr, tpr, roc_auc = dict(), dict(), dict()

for i in range(n_classes):
    fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], y_prob[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

plt.figure(figsize=(8, 6))
colors = ['blue', 'red', 'green', 'purple']
for i, color in zip(range(n_classes), colors):
    plt.plot(fpr[i], tpr[i], color=color, lw=2,
             label=f'ROC curve of class {class_names[i]} (area = {roc_auc[i]:.4f})')

plt.plot([0, 1], [0, 1], 'k--', lw=2)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', fontweight='bold')
plt.ylabel('True Positive Rate', fontweight='bold')
plt.title('Receiver Operating Characteristic (ROC) - SVM Fusion', fontsize=14, pad=15)
plt.legend(loc="lower right")
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.savefig("SVM_Fusion_ROC_Curve.png", dpi=300)
plt.close()

# --- DIAGRAM 3: t-SNE Feature Visualization ---
print("🌌 Squashing 3072-D space into 2D for t-SNE scatter plot...")
tsne = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000)
X_test_2D = tsne.fit_transform(X_test)

plt.figure(figsize=(10, 8))
scatter = plt.scatter(X_test_2D[:, 0], X_test_2D[:, 1], c=y_test, cmap='Set1', alpha=0.8, edgecolors='k')
plt.legend(handles=scatter.legend_elements()[0], labels=class_names, title="Tumor Classes")
plt.title('t-SNE Visualization of Fused 3072-D Feature Space', fontsize=14, pad=15)
plt.xlabel('t-SNE Dimension 1')
plt.ylabel('t-SNE Dimension 2')
plt.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig("SVM_Fusion_tSNE_Plot.png", dpi=300)
plt.close()

print("✅ PHASE 2 COMPLETE. All IEEE Diagrams saved to disk.")

⚔️ INITIATING PHASE 2: DECAPITATION & 3072-D FUSION ENGINE...
🪓 Decapitating DenseNet121 (Extracting 1024-D Vectors)...
🪓 Decapitating EfficientNet-B0 (Extracting 1280-D Vectors)...
🪓 Decapitating ViT-B/16 (Extracting 768-D Vectors)...
🧬 Extracting 3072-D Vectors for Training Data...
🧬 Extracting 3072-D Vectors for Testing Data...
✅ Extraction Complete. Fused Matrix Shape: (5712, 3072)

🧠 TRAINING THE MASTER SVM CLASSIFIER...

🏆 MASTER SVM ACCURACY: 99.47%

📊 GENERATING IEEE PUBLICATION DIAGRAMS...
🌌 Squashing 3072-D space into 2D for t-SNE scatter plot...
✅ PHASE 2 COMPLETE. All IEEE Diagrams saved to disk.


In [9]:
import pickle

# Save the SVM Master Classifier as .pkl to match the FastAPI backend
with open('master_svm_model.pkl', 'wb') as f:
    pickle.dump(svm, f)
print("✅ MASTER SVM SECURED AS .PKL.")

✅ MASTER SVM SECURED AS .PKL.
